# 🌿 FloraDB Starter — Load & Explore Houseplant Care

This notebook loads the **FloraDB free sample** (100 curated houseplants) and walks through:
1. Loading the CSV and previewing the schema
2. The honest quality flags (`care_confidence`, `toxicity_status`)
3. A **pet-safety filter** — plants explicitly safe for cats (never guessed)
4. Care-metric analysis (light vs watering)
5. Re-verifiable provenance via GBIF

The full dataset — 270 care plants, 891 ASPCA toxicity records, and a 20,000+ species taxonomic index — is at **[floradb](https://houseplants-botanical-floradb.pages.dev)**.

> Toxicity data is informational, not veterinary advice.


In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

# Find the sample CSV whether on Kaggle (/kaggle/input/...) or locally.
paths = glob.glob('/kaggle/input/**/floradb_sample.csv', recursive=True) + \
        glob.glob('**/floradb_sample.csv', recursive=True)
csv_path = paths[0]
df = pd.read_csv(csv_path)
print(f'{len(df)} plants from {df.family.nunique()} families')
df.head()

## Schema at a glance
Every plant has full care metrics; toxicity and enrichment are partial and **flagged**, so you can filter on quality.


In [ ]:
print('Columns:', list(df.columns))
print()
print('care_confidence:')
print(df.care_confidence.value_counts().to_string())
print()
print('toxicity_status (never guessed "safe"):')
print(df.toxicity_status.value_counts().to_string())

## Pet safety — plants explicitly safe for cats
FloraDB never guesses "safe": a plant only counts as cat-safe when its toxicity was **determined** (`toxicity_status != unknown`) and `is_toxic_to_cats == 0`.


In [ ]:
determined = df[df.toxicity_status != 'unknown']
cat_safe = determined[determined.is_toxic_to_cats == 0]
print(f'{len(cat_safe)} of {len(determined)} determined plants are safe for cats')
cat_safe[['common_name','scientific_name','family','toxicity_status']].head(10)

## Care metrics — light vs watering
Light is a category mapped to a standard Lux band; watering is an interval in days. Together they cluster plants into care regimes.


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for conf, sub in df.groupby('care_confidence'):
    ax.scatter(sub.max_lux, sub.watering_frequency_days, label=conf, alpha=0.7)
ax.set_xlabel('Max light (lux)'); ax.set_ylabel('Watering interval (days)')
ax.set_title('FloraDB care regimes — light vs watering'); ax.legend(title='care_confidence')
plt.tight_layout(); plt.show()

## Provenance — re-verify any record
Every plant carries its GBIF-accepted name and a `gbif_source_url` you can open to confirm the taxonomy.


In [ ]:
r = df.iloc[0]
print(f"{r.common_name} — {r.scientific_name} ({r.family})")
print(f"  light : {r.light_requirement_level} ({r.min_lux}-{r.max_lux} lux)")
print(f"  water : every {r.watering_frequency_days} days | {r.min_temp_celsius}-{r.max_temp_celsius}C | {r.ideal_humidity_percent}% humidity")
print(f"  toxic : dogs={r.is_toxic_to_dogs} cats={r.is_toxic_to_cats} [{r.toxicity_status}]")
print(f"  source: {r.gbif_source_url}")

---
**Get the full dataset:** [houseplants-botanical-floradb.pages.dev](https://houseplants-botanical-floradb.pages.dev) · Sample licensed CC-BY-NC-4.0.
